# (Optional) Test MCP Servers: With Metadata vs Legacy

동일한 질의를 두 MCP 서버에 보내 AI-Ready Data Lake의 효과를 비교합니다.

- `with-metadata` (`fhir-mcp-server`): 확장 컬럼명 + COMMENT 메타데이터 (`data` 네임스페이스)
- `legacy` (`fhir-mcp-legacy`): 축약 컬럼명, 메타데이터 없음 (`data_legacy` 네임스페이스)

## Step 1: Configuration

In [ ]:
import boto3, json, time, os

REGION = boto3.session.Session().region_name or os.environ.get("AWS_DEFAULT_REGION") or os.environ.get("AWS_REGION")
lambda_client = boto3.client("lambda", region_name=REGION)

LAMBDAS = {
    "with-metadata": "fhir-mcp-server",
    "legacy": "fhir-mcp-legacy",
}

def invoke_tool(func_name, tool_name, args):
    start = time.time()
    resp = lambda_client.invoke(FunctionName=func_name, Payload=json.dumps({"toolName": tool_name, "arguments": args}))
    result = json.loads(resp["Payload"].read())
    return result, time.time() - start

print("Ready. Lambda functions:", list(LAMBDAS.values()))


## Step 2: list_tables 비교

In [ ]:
for label, func in LAMBDAS.items():
    r, t = invoke_tool(func, "list_tables", {})
    data = json.loads(r.get("result", "{}")) if isinstance(r.get("result"), str) else r.get("result", {})
    tables = data.get("tables", [])
    print(f"[{label}] {len(tables)} tables ({t:.1f}s)")
    for tbl in tables[:3]:
        name = tbl.get("table", "")
        desc = tbl.get("description", "")
        cols = tbl.get("columns", [])
        print(f"  - {name}: {desc[:50] if desc else '(no description)'} | cols: {cols[:3]}")
    print()


## Step 3: get_table_schema 비교 (patient)

In [ ]:
for label, func in LAMBDAS.items():
    r, t = invoke_tool(func, "get_table_schema", {"table_name": "patient" if label == "with-metadata" else "tbl_01"})
    data = json.loads(r.get("result", "{}")) if isinstance(r.get("result"), str) else r.get("result", {})
    cols = data.get("columns", [])
    print(f"[{label}] schema ({t:.1f}s) — {len(cols)} columns")
    for c in cols[:5]:
        print(f"  {c.get('column_name','')}: {c.get('data_type','')} — {c.get('description','')[:40]}")
    print()


## Step 4: run_custom_query 비교 — 환자 수

In [ ]:
queries = {
    "with-metadata": "SELECT COUNT(*) as total FROM s3tablescatalog.`s3tablescatalog`.data.patient",
    "legacy": "SELECT COUNT(*) as total FROM s3tablescatalog.`s3tablescatalog`.data_legacy.tbl_01",
}

for label, func in LAMBDAS.items():
    r, t = invoke_tool(func, "run_custom_query", {"query": queries[label]})
    print(f"[{label}] ({t:.1f}s)")
    print(json.dumps(r.get("result", r), indent=2, ensure_ascii=False)[:300])
    print()


## Step 5: run_custom_query 비교 — 성별 분포

In [ ]:
queries = {
    "with-metadata": "SELECT gender, COUNT(*) as cnt FROM s3tablescatalog.`s3tablescatalog`.data.patient GROUP BY gender ORDER BY cnt DESC",
    "legacy": "SELECT c08, COUNT(*) as cnt FROM s3tablescatalog.`s3tablescatalog`.data_legacy.tbl_01 GROUP BY c08 ORDER BY cnt DESC",
}

for label, func in LAMBDAS.items():
    r, t = invoke_tool(func, "run_custom_query", {"query": queries[label]})
    print(f"[{label}] ({t:.1f}s)")
    print(json.dumps(r.get("result", r), indent=2, ensure_ascii=False)[:500])
    print()


## ✅ Summary

| 항목 | with-metadata (data) | legacy (data_legacy) |
|------|---------------------|-----------------------------|
| 테이블명 | `patient`, `encounter` | `tbl_01`, `tbl_02` |
| 컬럼명 | `gender`, `birth_date` | `c08`, `c09` |
| 메타데이터 | ✅ description, domain, code_values | ❌ 없음 |
| AI Text-to-SQL | ✅ 정확한 쿼리 생성 | ❌ 컬럼/테이블 추론 불가 |

AI-Ready Data Lake의 메타데이터가 Text-to-SQL 정확도에 미치는 영향을 확인할 수 있습니다.